# End-to-End Project: Sentiment Analysis Pipeline

An NLP project for text classification with multiple approaches.

## Project Overview

**Objective**: Build a sentiment analysis system to classify reviews as positive/negative.

**Skills Applied**:
- Text preprocessing and cleaning
- Multiple vectorization methods (TF-IDF, Count)
- Model comparison for text classification
- Pipeline design for NLP
- Error analysis and interpretability

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import re
from collections import Counter

# Sklearn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_recall_curve
)

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)

print("Libraries loaded successfully!")

## 1. Data Generation

We'll create a synthetic review dataset with sentiment labels.

In [ ]:
def generate_sentiment_data(n_samples=2000):
    """
    Generate synthetic sentiment data with realistic patterns.
    """
    np.random.seed(42)
    
    # Positive templates and phrases
    positive_templates = [
        "I absolutely loved this {product}! It was {adjective}.",
        "This {product} exceeded my expectations. {recommendation}",
        "Great {product}! The {feature} was {adjective}.",
        "Would definitely recommend this {product}. {positive_phrase}",
        "Amazing quality and excellent {feature}. {recommendation}",
        "Best {product} I've ever purchased. {adjective} experience!",
        "Love it! The {feature} is exactly what I needed. {positive_phrase}",
        "Outstanding {product} with {adjective} {feature}.",
        "Five stars! {positive_phrase} Very satisfied.",
        "Perfect {product}. {recommendation}"
    ]
    
    positive_adjectives = ['amazing', 'wonderful', 'fantastic', 'excellent', 'superb', 
                          'incredible', 'outstanding', 'perfect', 'brilliant', 'great']
    positive_phrases = [
        "Would buy again.", "Highly recommend!", "Worth every penny.",
        "Couldn't be happier.", "Exceeded expectations.", "Best purchase ever.",
        "Simply amazing.", "Love everything about it."
    ]
    recommendations = [
        "Highly recommend to everyone.", "Must buy!", "You won't regret it.",
        "Get it now!", "Best decision I made."
    ]
    
    # Negative templates and phrases
    negative_templates = [
        "I was disappointed with this {product}. It was {adjective}.",
        "This {product} did not meet my expectations. {negative_phrase}",
        "Poor quality {product}. The {feature} was {adjective}.",
        "Would not recommend this {product}. {negative_phrase}",
        "Terrible experience with this {product}. {negative_phrase}",
        "Waste of money. The {product} was {adjective}.",
        "Very unhappy with this purchase. {negative_phrase}",
        "Awful {product}. {feature} doesn't work properly.",
        "One star. {negative_phrase} Very disappointed.",
        "Regret buying this {product}. {adjective} quality."
    ]
    
    negative_adjectives = ['terrible', 'awful', 'horrible', 'disappointing', 'poor', 
                          'bad', 'worst', 'useless', 'defective', 'broken']
    negative_phrases = [
        "Complete waste of money.", "Don't buy this.", "Returning immediately.",
        "Very frustrated.", "Total disappointment.", "Worst purchase ever.",
        "Stay away.", "Save your money."
    ]
    
    # Common elements
    products = ['product', 'item', 'purchase', 'device', 'tool', 'gadget', 'thing']
    features = ['quality', 'design', 'performance', 'durability', 'functionality', 
                'build', 'material', 'finish']
    
    reviews = []
    labels = []
    
    n_positive = n_samples // 2
    n_negative = n_samples - n_positive
    
    # Generate positive reviews
    for _ in range(n_positive):
        template = np.random.choice(positive_templates)
        review = template.format(
            product=np.random.choice(products),
            adjective=np.random.choice(positive_adjectives),
            feature=np.random.choice(features),
            positive_phrase=np.random.choice(positive_phrases),
            recommendation=np.random.choice(recommendations)
        )
        reviews.append(review)
        labels.append(1)
    
    # Generate negative reviews
    for _ in range(n_negative):
        template = np.random.choice(negative_templates)
        review = template.format(
            product=np.random.choice(products),
            adjective=np.random.choice(negative_adjectives),
            feature=np.random.choice(features),
            negative_phrase=np.random.choice(negative_phrases)
        )
        reviews.append(review)
        labels.append(0)
    
    # Shuffle
    indices = np.random.permutation(len(reviews))
    reviews = [reviews[i] for i in indices]
    labels = [labels[i] for i in indices]
    
    return pd.DataFrame({'text': reviews, 'sentiment': labels})


# Generate dataset
df = generate_sentiment_data(3000)

print(f"Dataset shape: {df.shape}")
print(f"\nSentiment distribution:")
print(df['sentiment'].value_counts())
print("\nSample reviews:")
df.head()

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['sentiment'].value_counts()
axes[0].bar(['Negative', 'Positive'], counts.values, color=['#e74c3c', '#2ecc71'])
axes[0].set_ylabel('Count')
axes[0].set_title('Sentiment Distribution')

# Review length distribution
df['text_length'] = df['text'].apply(len)
axes[1].hist(df[df['sentiment']==1]['text_length'], bins=20, alpha=0.7, label='Positive')
axes[1].hist(df[df['sentiment']==0]['text_length'], bins=20, alpha=0.7, label='Negative')
axes[1].set_xlabel('Review Length (characters)')
axes[1].set_ylabel('Count')
axes[1].set_title('Review Length by Sentiment')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Text Preprocessing

In [ ]:
class TextPreprocessor:
    """
    Text preprocessing utilities for NLP.
    """
    
    # Simple stopwords list
    STOPWORDS = {'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been',
                 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
                 'would', 'could', 'should', 'may', 'might', 'must', 'can',
                 'to', 'of', 'in', 'for', 'on', 'with', 'at', 'by', 'from',
                 'as', 'into', 'through', 'during', 'before', 'after', 'above',
                 'below', 'between', 'under', 'again', 'further', 'then', 'once',
                 'and', 'but', 'or', 'nor', 'so', 'yet', 'both', 'either', 'neither',
                 'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'she', 'it',
                 'they', 'them', 'their', 'this', 'that', 'these', 'those'}
    
    def __init__(self, lowercase=True, remove_punctuation=True, 
                 remove_stopwords=True, min_word_length=2):
        self.lowercase = lowercase
        self.remove_punctuation = remove_punctuation
        self.remove_stopwords = remove_stopwords
        self.min_word_length = min_word_length
    
    def clean(self, text):
        """Clean and preprocess text."""
        # Lowercase
        if self.lowercase:
            text = text.lower()
        
        # Remove punctuation
        if self.remove_punctuation:
            text = re.sub(r'[^\w\s]', ' ', text)
        
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Remove stopwords and short words
        if self.remove_stopwords:
            words = text.split()
            words = [w for w in words 
                    if w not in self.STOPWORDS and len(w) >= self.min_word_length]
            text = ' '.join(words)
        
        return text
    
    def transform(self, texts):
        """Transform a list of texts."""
        return [self.clean(text) for text in texts]


# Create preprocessor
preprocessor = TextPreprocessor()

# Test preprocessing
sample_text = "I absolutely LOVED this product! It was amazing."
print(f"Original: {sample_text}")
print(f"Cleaned:  {preprocessor.clean(sample_text)}")

In [ ]:
# Apply preprocessing to dataset
df['text_clean'] = preprocessor.transform(df['text'].tolist())

# Show examples
print("Sample processed reviews:")
for i in range(3):
    print(f"\nOriginal: {df['text'].iloc[i]}")
    print(f"Cleaned:  {df['text_clean'].iloc[i]}")

In [ ]:
# Word frequency analysis
def get_word_frequencies(texts, n_top=20):
    """Get word frequencies from a list of texts."""
    all_words = ' '.join(texts).split()
    return Counter(all_words).most_common(n_top)

# Get word frequencies for each sentiment
positive_words = get_word_frequencies(df[df['sentiment']==1]['text_clean'].tolist())
negative_words = get_word_frequencies(df[df['sentiment']==0]['text_clean'].tolist())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Positive words
words, counts = zip(*positive_words)
axes[0].barh(list(words)[::-1], list(counts)[::-1], color='#2ecc71')
axes[0].set_xlabel('Frequency')
axes[0].set_title('Top Words in Positive Reviews')

# Negative words
words, counts = zip(*negative_words)
axes[1].barh(list(words)[::-1], list(counts)[::-1], color='#e74c3c')
axes[1].set_xlabel('Frequency')
axes[1].set_title('Top Words in Negative Reviews')

plt.tight_layout()
plt.show()

## 3. Data Splitting and Vectorization

In [ ]:
# Split data
X = df['text_clean']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Compare vectorization methods
vectorizers = {
    'CountVectorizer': CountVectorizer(max_features=5000, ngram_range=(1, 2)),
    'TfidfVectorizer': TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
}

# Test vectorizers
for name, vec in vectorizers.items():
    X_vec = vec.fit_transform(X_train)
    print(f"{name}: {X_vec.shape[1]} features")
    print(f"  Sample feature names: {vec.get_feature_names_out()[:5]}...")

## 4. Model Comparison

In [ ]:
# Define model pipelines
pipelines = {
    'Naive Bayes + Count': Pipeline([
        ('vectorizer', CountVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', MultinomialNB())
    ]),
    'Naive Bayes + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', MultinomialNB())
    ]),
    'Logistic Regression + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'SVM + TF-IDF': Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('classifier', LinearSVC(random_state=42))
    ])
}

# Train and evaluate
results = {}

for name, pipeline in pipelines.items():
    print(f"Training {name}...")
    
    # Cross-validation
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    
    # Fit on full training set
    pipeline.fit(X_train, y_train)
    
    # Test predictions
    y_pred = pipeline.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    
    results[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'test_accuracy': test_acc,
        'pipeline': pipeline
    }
    
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    print(f"  Test Accuracy: {test_acc:.4f}\n")

In [ ]:
# Results comparison
results_df = pd.DataFrame({
    'Model': results.keys(),
    'CV Accuracy': [r['cv_mean'] for r in results.values()],
    'CV Std': [r['cv_std'] for r in results.values()],
    'Test Accuracy': [r['test_accuracy'] for r in results.values()]
}).sort_values('Test Accuracy', ascending=False)

print("=== Model Comparison ===")
print(results_df.to_string(index=False))

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(results))
width = 0.35

cv_scores = [results[m]['cv_mean'] for m in results]
test_scores = [results[m]['test_accuracy'] for m in results]

bars1 = ax.bar(x - width/2, cv_scores, width, label='CV Accuracy', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test Accuracy', alpha=0.8)

ax.set_ylabel('Accuracy')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(list(results.keys()), rotation=15, ha='right')
ax.legend()
ax.set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

## 5. Best Model Analysis

In [ ]:
# Select best model
best_model_name = max(results, key=lambda x: results[x]['test_accuracy'])
best_pipeline = results[best_model_name]['pipeline']

print(f"Best Model: {best_model_name}")

# Predictions
y_pred = best_pipeline.predict(X_test)

print("\n" + "="*50)
print("Classification Report:")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Get feature importance from Logistic Regression
lr_pipeline = results['Logistic Regression + TF-IDF']['pipeline']
vectorizer = lr_pipeline.named_steps['vectorizer']
classifier = lr_pipeline.named_steps['classifier']

# Get feature names and coefficients
feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

# Create DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient')

# Get top positive and negative features
n_features = 15
top_positive = importance_df.tail(n_features)
top_negative = importance_df.head(n_features)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Positive features
axes[0].barh(top_positive['feature'], top_positive['coefficient'], color='#2ecc71')
axes[0].set_xlabel('Coefficient')
axes[0].set_title('Words Associated with Positive Sentiment')

# Negative features
axes[1].barh(top_negative['feature'], top_negative['coefficient'], color='#e74c3c')
axes[1].set_xlabel('Coefficient')
axes[1].set_title('Words Associated with Negative Sentiment')

plt.tight_layout()
plt.show()

## 7. Error Analysis

In [ ]:
# Find misclassified examples
X_test_list = X_test.tolist()
y_test_list = y_test.tolist()
y_pred_list = y_pred.tolist()

misclassified = []
for i in range(len(y_test_list)):
    if y_test_list[i] != y_pred_list[i]:
        misclassified.append({
            'text': X_test_list[i],
            'true': 'Positive' if y_test_list[i] == 1 else 'Negative',
            'predicted': 'Positive' if y_pred_list[i] == 1 else 'Negative'
        })

print(f"Total misclassified: {len(misclassified)} out of {len(y_test_list)} ({len(misclassified)/len(y_test_list)*100:.1f}%)")

print("\n=== Sample Misclassified Reviews ===")
for i, item in enumerate(misclassified[:5]):
    print(f"\n{i+1}. True: {item['true']}, Predicted: {item['predicted']}")
    print(f"   Text: {item['text'][:100]}..." if len(item['text']) > 100 else f"   Text: {item['text']}")

In [ ]:
# Analyze error patterns
false_positives = [m for m in misclassified if m['true'] == 'Negative']
false_negatives = [m for m in misclassified if m['true'] == 'Positive']

print(f"False Positives (predicted positive, actually negative): {len(false_positives)}")
print(f"False Negatives (predicted negative, actually positive): {len(false_negatives)}")

## 8. Inference Pipeline

In [ ]:
class SentimentAnalyzer:
    """
    Production-ready sentiment analysis pipeline.
    """
    
    def __init__(self, pipeline, preprocessor=None):
        self.pipeline = pipeline
        self.preprocessor = preprocessor or TextPreprocessor()
        self.labels = ['Negative', 'Positive']
    
    def analyze(self, text):
        """
        Analyze sentiment of a single text.
        """
        # Preprocess
        clean_text = self.preprocessor.clean(text)
        
        # Predict
        prediction = self.pipeline.predict([clean_text])[0]
        
        # Get confidence if available
        confidence = None
        if hasattr(self.pipeline, 'predict_proba'):
            proba = self.pipeline.predict_proba([clean_text])[0]
            confidence = proba[prediction]
        elif hasattr(self.pipeline.named_steps['classifier'], 'decision_function'):
            decision = self.pipeline.decision_function([clean_text])[0]
            confidence = abs(decision)
        
        return {
            'text': text,
            'sentiment': self.labels[prediction],
            'sentiment_score': int(prediction),
            'confidence': float(confidence) if confidence is not None else None
        }
    
    def analyze_batch(self, texts):
        """
        Analyze sentiment of multiple texts.
        """
        return [self.analyze(text) for text in texts]


# Create analyzer
analyzer = SentimentAnalyzer(best_pipeline, preprocessor)

# Test
test_reviews = [
    "This product is amazing! Best purchase ever.",
    "Terrible quality. Complete waste of money.",
    "It's okay, nothing special.",
    "Absolutely love it! Highly recommend!"
]

print("=== Sentiment Analysis Results ===")
for result in analyzer.analyze_batch(test_reviews):
    print(f"\nText: {result['text']}")
    print(f"Sentiment: {result['sentiment']}")
    if result['confidence']:
        print(f"Confidence: {result['confidence']:.4f}")

## 9. Model Serialization

In [ ]:
import joblib

# Save model
model_path = Path('./models')
model_path.mkdir(exist_ok=True)

model_data = {
    'pipeline': best_pipeline,
    'model_name': best_model_name,
    'metrics': {
        'test_accuracy': results[best_model_name]['test_accuracy'],
        'cv_accuracy': results[best_model_name]['cv_mean']
    }
}

joblib.dump(model_data, model_path / 'sentiment_model.joblib')
print(f"Model saved to {model_path / 'sentiment_model.joblib'}")

In [ ]:
# Load and verify
loaded_data = joblib.load(model_path / 'sentiment_model.joblib')
loaded_pipeline = loaded_data['pipeline']

# Test loaded model
loaded_analyzer = SentimentAnalyzer(loaded_pipeline, preprocessor)
test_result = loaded_analyzer.analyze("This is a great product!")

print(f"Model: {loaded_data['model_name']}")
print(f"Metrics: {loaded_data['metrics']}")
print(f"\nTest prediction: {test_result['sentiment']}")

## Summary

### Skills Demonstrated

- **Text preprocessing**: Cleaning, stopword removal, normalization
- **Vectorization comparison**: Count vs TF-IDF
- **Model comparison**: Naive Bayes, Logistic Regression, SVM
- **Pipeline design**: End-to-end sklearn pipelines
- **Feature importance**: Understanding model decisions
- **Error analysis**: Identifying misclassification patterns
- **Production inference**: Clean API for deployment

### Key Takeaways

1. Text preprocessing significantly impacts model performance
2. TF-IDF often outperforms simple counts for classification
3. Feature importance reveals what the model learned
4. Error analysis helps identify model weaknesses
5. Pipelines simplify deployment and ensure consistency